## 🌱 Predicting Optimal Fertilizers: A Structured Approach

In [1]:
# 📦 Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 🧠 Machine learning and preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

### 1. 📁 Understanding the Data Files

Files Overview:
* train.csv: Contains features and the target variable (Fertilizer Name).

* test.csv: Contains features; predictions are made for this set.

* sample_submission.csv: Provides the submission format.

In [2]:
# 🗂️ Load training and test datasets
train = pd.read_csv('/kaggle/input/playground-series-s5e6/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e6/test.csv')
sample_submission = pd.read_csv('/kaggle/input/playground-series-s5e6/sample_submission.csv')

In [3]:
# ✅ Display data shape
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Sample Submission shape: {sample_submission.shape}")

# 📝 Preview first few rows of each file
print("\nTrain sample:")
print(train.head())
print("\nTest sample:")
print(test.head())
print("\nSample Submission sample:")

Train shape: (750000, 10)
Test shape: (250000, 9)
Sample Submission shape: (250000, 2)

Train sample:
   id  Temparature  Humidity  Moisture Soil Type  Crop Type  Nitrogen  \
0   0           37        70        36    Clayey  Sugarcane        36   
1   1           27        69        65     Sandy    Millets        30   
2   2           29        63        32     Sandy    Millets        24   
3   3           35        62        54     Sandy     Barley        39   
4   4           35        58        43       Red      Paddy        37   

   Potassium  Phosphorous Fertilizer Name  
0          4            5           28-28  
1          6           18           28-28  
2         12           16        17-17-17  
3         12            4        10-26-26  
4          2           16             DAP  

Test sample:
       id  Temparature  Humidity  Moisture Soil Type    Crop Type  Nitrogen  \
0  750000           31        70        52     Sandy        Wheat        34   
1  750001           27 

In [4]:
# 👀 Overview of training data
print("\n🔍 Train Data Info:")
print(train.info())


🔍 Train Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   id               750000 non-null  int64 
 1   Temparature      750000 non-null  int64 
 2   Humidity         750000 non-null  int64 
 3   Moisture         750000 non-null  int64 
 4   Soil Type        750000 non-null  object
 5   Crop Type        750000 non-null  object
 6   Nitrogen         750000 non-null  int64 
 7   Potassium        750000 non-null  int64 
 8   Phosphorous      750000 non-null  int64 
 9   Fertilizer Name  750000 non-null  object
dtypes: int64(7), object(3)
memory usage: 57.2+ MB
None


### 2. 🔍 Data Preprocessing & Feature Engineering

#### Handling Missing Values:

In [5]:
# 🧼 Check for missing values
print("\n❓ Missing values in Train:")
print(train.isnull().sum())


❓ Missing values in Train:
id                 0
Temparature        0
Humidity           0
Moisture           0
Soil Type          0
Crop Type          0
Nitrogen           0
Potassium          0
Phosphorous        0
Fertilizer Name    0
dtype: int64


In [6]:
# 🧪 Unique values in categorical features
print("\n🧬 Unique values in 'Soil Type':", train['Soil Type'].unique())
print("🧬 Unique values in 'Crop Type':", train['Crop Type'].unique())
print("🎯 Unique values in 'Fertilizer Name':", train['Fertilizer Name'].unique())


🧬 Unique values in 'Soil Type': ['Clayey' 'Sandy' 'Red' 'Loamy' 'Black']
🧬 Unique values in 'Crop Type': ['Sugarcane' 'Millets' 'Barley' 'Paddy' 'Pulses' 'Tobacco' 'Ground Nuts'
 'Maize' 'Cotton' 'Wheat' 'Oil seeds']
🎯 Unique values in 'Fertilizer Name': ['28-28' '17-17-17' '10-26-26' 'DAP' '20-20' '14-35-14' 'Urea']


#### Encoding Categorical Variables:

In [7]:
# 🔁 Copy data to avoid mutation
train_data = train.copy()
test_data = test.copy()

In [8]:
# 🔣 Encode categorical features
cat_features = ['Soil Type', 'Crop Type']
label_encoders = {}

In [9]:
for col in cat_features:
    le = LabelEncoder()
    train_data[col] = le.fit_transform(train_data[col])
    test_data[col] = le.transform(test_data[col])  # Use same encoding
    label_encoders[col] = le  # Store for possible inverse_transform

In [10]:
# 🎯 Encode target label
fertilizer_le = LabelEncoder()
train_data['Fertilizer Name'] = fertilizer_le.fit_transform(train_data['Fertilizer Name'])

#### Feature Scaling:

In [11]:
# 🧪 Feature matrix and target vector
X = train_data.drop(columns=['id', 'Fertilizer Name'])
y = train_data['Fertilizer Name']
X_test = test_data.drop(columns=['id'])

In [12]:
# 🔍 Scaling features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

In [13]:
# 🚀 Print status
print("🔢 Classes:", fertilizer_le.classes_)

🔢 Classes: ['10-26-26' '14-35-14' '17-17-17' '20-20' '28-28' 'DAP' 'Urea']


### 4. 🤖 Model Selection & Training

#### Model 1: Logistic Regression

In [14]:
# 📊 Custom MAP@3 scorer
def mapk(actual, predicted, k=3):
    """
    Computes the mean average precision at k.
    actual: list of actual labels (ints)
    predicted: list of predicted label lists (ints)
    """
    score = 0.0
    for a, p in zip(actual, predicted):
        if a in p[:k]:
            score += 1.0 / (p.index(a) + 1)
    return score / len(actual)

# 🧪 Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# 🔁 Train a simple baseline model
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

# 🔮 Predict top 3 classes
val_probs = logreg.predict_proba(X_val)
top_3_preds = np.argsort(val_probs, axis=1)[:, -3:][:, ::-1]

# 🎯 Evaluate MAP@3
map3_score = mapk(y_val.tolist(), top_3_preds.tolist(), k=3)
print(f"📊 Logistic Regression MAP@3 score: {map3_score:.4f}")

📊 Logistic Regression MAP@3 score: 0.2846


#### Model 2: Random Forest

In [15]:
# 🌳🌳🌳 Train a Random Forest 🌳🌳🌳
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 🔮 Predict probabilities
val_probs_rf = rf_model.predict_proba(X_val)
top_3_preds_rf = np.argsort(val_probs_rf, axis=1)[:, -3:][:, ::-1]

# 🎯 Evaluate MAP@3
map3_rf = mapk(y_val.tolist(), top_3_preds_rf.tolist(), k=3)
print(f"🌳 Random Forest MAP@3 score: {map3_rf:.4f}")

🌳 Random Forest MAP@3 score: 0.2929


#### Model 3: XGBoost Model

In [16]:
# ⚡ Train an XGBoost Classifier
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    objective='multi:softprob',
    num_class=len(fertilizer_le.classes_),
    eval_metric='mlogloss',
    use_label_encoder=False,
    n_jobs=-1,
    random_state=42
)
xgb_model.fit(X_train, y_train)

# 🔮 Predict probabilities
val_probs_xgb = xgb_model.predict_proba(X_val)
top_3_preds_xgb = np.argsort(val_probs_xgb, axis=1)[:, -3:][:, ::-1]

# 🎯 Evaluate MAP@3
map3_xgb = mapk(y_val.tolist(), top_3_preds_xgb.tolist(), k=3)
print(f"⚡ XGBoost MAP@3 score: {map3_xgb:.4f}")

⚡ XGBoost MAP@3 score: 0.3193


### 5. 📤 Preparing the Submission

In [17]:
# 🔮 Predict probabilities on test set
test_probs = xgb_model.predict_proba(X_test_scaled)

# 📌 Get top 3 predictions for each row
top_3_test_preds = np.argsort(test_probs, axis=1)[:, -3:][:, ::-1]

# 🔁 Convert indices to fertilizer names
top_3_labels = fertilizer_le.inverse_transform(top_3_test_preds.ravel()).reshape(top_3_test_preds.shape)

# 📝 Build submission DataFrame
submission = pd.DataFrame({
    'id': test['id'],
    'Fertilizer Name': [' '.join(row) for row in top_3_labels]
})

# 💾 Save to CSV
submission.to_csv('submission.csv', index=False)

# ✅ Display first few rows of submission
print(submission.head())

       id             Fertilizer Name
0  750000             DAP 28-28 20-20
1  750001     17-17-17 20-20 10-26-26
2  750002     14-35-14 10-26-26 28-28
3  750003  14-35-14 17-17-17 10-26-26
4  750004     20-20 10-26-26 17-17-17
